# S5.G3-3 · G3 五臂对比 FedAds：配对比较

> 评估口径 v1：两层重抽样（seed × 用户簇）2,000 次、保守可检出差异、四类结论；次要对比 Holm 校正按 S6.P3 完整家族 9 项计。开发证据，封存段未抽取；不代表汇丰真实效果。

In [1]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_g3_fedads_run.yaml"
run = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
PRE_PATH, ADD_PATH = REPO_ROOT / run["preregistration"], REPO_ROOT / run["addendum"]
pre = yaml.safe_load(PRE_PATH.read_text(encoding="utf-8"))
add = yaml.safe_load(ADD_PATH.read_text(encoding="utf-8"))
ad, fa = pre["freeze"]["arm_details"], add["arm_details"]


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + sha(CONFIG_PATH))
print("preregistration:", PRE_PATH.relative_to(REPO_ROOT), "sha256:" + sha(PRE_PATH), "| status:", pre["status"])
print("addendum:", ADD_PATH.relative_to(REPO_ROOT), "sha256:" + sha(ADD_PATH), "| status:", add["status"])
print("seed:", run["seed"], "| seeds:", run["seeds"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
print("step:", run["step_id"], "|", run["purpose"])
mismatch = [p for p, h in add["code_sha256"].items() if hashlib.sha256((REPO_ROOT / p).read_bytes()).hexdigest() != h]
print("冻结代码指纹核对：", "一致" if not mismatch else "不一致 " + str(mismatch))
assert pre["status"] == "frozen" and add["status"] == "frozen" and not mismatch
assert sha(PRE_PATH, None) == add["parent_sha256"]
PRIV = pathlib.Path(os.environ[run["private_root_env"]])
SEED0 = run["seeds"][0]
MB, NJ = ad["xgboost_max_bin"], ad["n_jobs"]

config: modules/m5_modeling/configs/s5_g3_fedads_run.yaml sha256:56f1fdbd6540
preregistration: modules/m6_evaluation/configs/s6_p3_g3_preregistration.yaml sha256:c0a4bbfc9709 | status: frozen
addendum: modules/m6_evaluation/configs/s6_p4_fedads_addendum.yaml sha256:462b0a74e10b | status: frozen
seed: 11 | seeds: [11, 22, 33, 44, 55, 66]
git: 540d5b9（工作区含本步未提交改动）
step: S5.G3 | 按冻结的 S6.P3 与 S6.P4 在 FedAds 上训练 L0、L1（k=5/10/20）、B_T1、A（SGB）、集中式代理 L4 与阴性对照 NC，并做配对比较
冻结代码指纹核对： 一致


In [2]:
from modules.m6_evaluation.components import metrics as MT
from modules.m6_evaluation.components import paired_eval as PE

arr = np.load(PRIV / run["arrays_file"])
local = np.load(PRIV / run["local_scores_file"])
sgb = np.load(PRIV / run["sgb_scores_file"])
TUNING = json.loads((PRIV / run["tuning_file"]).read_text(encoding="utf-8"))
y = arr["dev_eval__y"]
clusters = arr["dev_eval__cust"]
S = len(run["seeds"])


def as_matrix(v):
    v = np.asarray(v)
    return np.tile(v, (S, 1)) if v.ndim == 1 else v


SC = {a: as_matrix(local[a]) for a in local.files}
SC["A"] = as_matrix(sgb["dev_eval"])
SC["B_best"] = SC[TUNING["B_best"]]
n_pos, base = int(y.sum()), float(y.mean())
rows = []
for a in run["report_arms"]:
    m = SC[a]
    rows.append({"arm": a, "auroc": float(np.mean([MT.auc(y, s) for s in m])),
                 "lift_at_k": float(np.mean([MT.lift_at_k(y, s, run["top_k_share"]) for s in m])),
                 "recall_at_k": float(np.mean([MT.recall_at_k(y, s, run["top_k_share"]) for s in m])),
                 "seed_sd": float(np.std([MT.auc(y, s) for s in m]))})
arms = pd.DataFrame(rows)
arms["precision_at_k"] = arms["lift_at_k"] * base
print("开发评估集：%d 行、%d 位用户、%d 个正例行（正例率 %.4f）" % (len(y), len(np.unique(clusters)), n_pos, base))
print(arms.round(run["round_auc"]).to_string(index=False))

开发评估集：129489 行、91967 位用户、848 个正例行（正例率 0.0065）
   arm  auroc  lift_at_k  recall_at_k  seed_sd  precision_at_k
    L0 0.5935     1.7690       0.1769   0.0000          0.0116
 L1_k5 0.5849     1.6511       0.1651   0.0000          0.0108
L1_k10 0.5836     1.7690       0.1769   0.0000          0.0116
L1_k20 0.5949     1.6157       0.1616   0.0000          0.0106
  B_T1 0.6265     2.0638       0.2064   0.0029          0.0135
     A 0.6184     1.9931       0.1993   0.0000          0.0131
    L4 0.6166     1.9931       0.1993   0.0000          0.0131
    NC 0.5933     1.6727       0.1673   0.0030          0.0110


**读法**：开发评估集 129,489 行、91,967 位用户、848 个正例行（正例率 0.65%），达到预注册目标 700。转化预测本身难，各臂绝对 AUROC 只有 0.58–0.63。L0 0.593；三种 L1 在 0.584–0.595 之间，没有高于 L0；B_T1 0.626、A 0.618、L4 0.617；阴性对照 NC 0.593 与 L0 相同。前 10% 名单的命中率：L0 1.16%，B_T1 1.35%，A 1.31%。

In [3]:
rng = np.random.default_rng(run["seed"])
alpha, power = run["alpha"], run["power"]
delta = pre["primary"]["delta_of_interest"]
planned = [("primary", pre["primary"]["contrast"])] + [("secondary", c) for c in pre["secondary_contrasts"]]
planned.append(("check", ["A", "L4"]))
skip = [list(c) for c in fa["secondary_contrasts_skipped"]]
contrasts = [(k, c) for k, c in planned if c not in skip and c[0] in SC and c[1] in SC]
print("本数据集不做的对比（S6.P4 F-6）：", [" − ".join(c) for c in skip])
out = []
for kind, (a, b) in contrasts:
    res = PE.paired_bootstrap(y, SC[a], SC[b], clusters, run["n_bootstrap"], alpha, rng)
    gap = PE.detectable_gap(res, n_pos, base, alpha, power, PE.MDE_CONSERVATIVE)
    out.append({"kind": kind, "contrast": "%s − %s" % (a, b), "diff": res["diff"], "ci_low": res["ci_low"],
                "ci_high": res["ci_high"], "p_value": res["p_value"], "detectable_gap": gap,
                "verdict": PE.verdict(res, gap, delta), "old_rule_pass": res["old_rule_pass"],
                "estimate_correlation": res["estimate_correlation"]})
con = pd.DataFrame(out)
sec = con["kind"] == "secondary"
order = con.loc[sec, "p_value"].sort_values().index
holm, running = {}, 0.0
for rank, idx in enumerate(order):                       # Holm 按 S6.P3 完整次要家族计（S6.P4 holm_family_size），偏保守
    running = max(running, min(1.0, (fa["holm_family_size"] - rank) * con.loc[idx, "p_value"]))
    holm[idx] = running
con["p_holm"] = pd.Series(holm)
print(con.drop(columns=["estimate_correlation"]).round(run["round_auc"]).to_string(index=False))
res_dir = REPO_ROOT / run["results_dir"]
res_dir.mkdir(parents=True, exist_ok=True)
arms.to_csv(res_dir / "arms_dev_eval.csv", index=False)
con.to_csv(res_dir / "contrasts_dev_eval.csv", index=False)
cost = json.loads((PRIV / run["timing_file"]).read_text(encoding="utf-8"))
pd.DataFrame([{"n_rows": len(y), "n_users": len(np.unique(clusters)), "n_positive_rows": n_pos,
               "b_best": TUNING["B_best"], "sgb_fit_seconds": cost["fit_seconds"]}]).to_csv(
    res_dir / "run_summary.csv", index=False)
print("已写出公开汇总：", sorted(p.name for p in res_dir.iterdir()))

本数据集不做的对比（S6.P4 F-6）： ['B_T2 − L1_k10', 'A_secure − A']


     kind        contrast    diff  ci_low  ci_high  p_value  detectable_gap       verdict  old_rule_pass  p_holm
  primary      A − L1_k10  0.0347  0.0199   0.0495    0.000          0.0229        可分辨的提升          False     NaN
secondary          A − L0  0.0249  0.0147   0.0354    0.000          0.0162        可分辨的提升          False   0.000
secondary     L1_k10 − L0 -0.0098 -0.0232   0.0041    0.154          0.0206       未达可检出规模          False   0.770
secondary   B_T1 − L1_k10  0.0428  0.0266   0.0592    0.000          0.0246        可分辨的提升           True   0.000
secondary      A − B_best -0.0081 -0.0168   0.0001    0.055          0.0130 在预声明尺度上无可分辨差异          False   0.371
secondary         NC − L0 -0.0002 -0.0063   0.0061    0.938          0.0096 在预声明尺度上无可分辨差异          False   1.000
secondary  L1_k5 − L1_k10  0.0013 -0.0085   0.0106    0.802          0.0141 在预声明尺度上无可分辨差异          False   1.000
secondary L1_k20 − L1_k10  0.0113 -0.0001   0.0216    0.053          0.0169 在预声明尺度上无可分辨差异       

**读法**：

- **主对比 A − L1_k10 = +0.035 [0.020, 0.050]，可分辨的提升**（可检出差异 0.023）。
- L1 没有增量：L1_k10 − L0 = −0.010 [−0.023, 0.004]，区间含 0，可检出差异 0.021 略大于关心尺度，按口径记为「未达可检出规模」。
- 单向候选 B_T1 拿到全部联邦增益：B_T1 − L1_k10 = +0.043；A − B_T1 = −0.008 [−0.017, 0.000]，区间含 0，B_T1 数值上还略高于 A。
- 阴性对照 NC − L0 ≈ 0（区间含 0）：增益来自两方信息互补，不是容量或调参。无损性核对 A − L4 = +0.002 [−0.005, 0.009]。
- 被动方只有 5 个低基数列，B_T1 的叶子编码几乎等于把这 5 列原样发给内地（S7.G1 的结论在这里更强）。

## 结论

1. **自然切分上复现了 Santander 构造切分的定性结论**：联邦（A）相对「本地模型 + 对方 k-匿名统计」（L1）有可分辨的提升（+0.035），L1 相对本地单边没有增量。
2. **增益幅度比构造切分小**：A − L0 在 FedAds 为 +0.025，Santander 为 +0.044；构造切分把香港侧设计成信息丰富的一方，增益偏大的担心部分成立。
3. **单向形态（B_T1）与双向（A）分不开**，但它是把被动方特征几乎原样发给主动方——精度接近不等于披露更少（S7.G1）。
4. 阴性对照与无损性核对均通过。
5. **边界**：公开广告转化数据，训练只取约 15 万行（S6.P4 F-4），绝对 AUROC 低；不代表汇丰真实效果，不得换算成汇丰的命中率或收益；技术侧自评，未经法务复核（W-001）。